# 00 — Patent Filter: eVTOL Architecture Classes (Preliminary Analysis v2)

Multi-class architecture labelling experiment for DINOv2 (replaces the earlier
binary shrouded-vs-open-rotor experiment).

Two source Excel files are involved:

- `paths.architecture_excel` (`Custom_eVTOL_Dataset_100.xlsx`, sheet `Dataset`) —
  the curated 100-patent set with the `Architecture Category` label. Its own
  `PDF Link` column is a placeholder with **no real hyperlinks**.
- `paths.excel` (`1639__dataset_08_06_26.xlsx`) — the master PatSeer export with
  real `PDF Link` hyperlinks for ~1639 patents (98 of our 100 curated patents
  are present in it; 2 are missing entirely and get flagged below).

This notebook merges the two on `Record Number` to attach a real `pdf_url` to
each curated, labelled patent, then writes the two artifacts every downstream
notebook expects:

- `logs/selected_patents.json` — read by `get_subset()` (`subset.mode: "selected"`)
  to restrict `01_extraction` to exactly these patents.
- `experiment/selected_patents.csv` — read by `processor.organize_processed()`
  to know which `final/<category>/` folder each patent's image belongs in.

| Architecture Category (Excel) | category (slug) | Count |
|---|---|---|
| Compound Helicopter | compound_helicopter | 25 |
| Ducted Fan Vectored Thrust | ducted_fan_vectored_thrust | 25 |
| Lift + Cruise | lift_cruise | 25 |
| Multirotor | multirotor | 25 |


In [1]:
import sys; sys.path.insert(0, "..")
import pandas as pd
from pathlib import Path
from src.config_loader import load_config

cfg = load_config()

# Curated set: Architecture Category labels (real PDF links are NOT usable here)
arch_df = pd.read_excel(cfg["paths"]["architecture_excel"], sheet_name="Dataset",
                         dtype={"Record Number": str})
print(f"Curated/labelled patents : {len(arch_df)}")
print(arch_df["Architecture Category"].value_counts().to_string())

# Master export: real PDF Link hyperlinks, used downstream by 01_extraction
from src.patents import load_patents
master_df, master_missing_df = load_patents(cfg)
print(f"\nMaster export — valid PDF URL: {len(master_df)}, missing URL: {len(master_missing_df)}")


Curated/labelled patents : 128
Architecture Category
Compound Helicopter           32
Ducted Fan Vectored Thrust    32
Lift + Cruise                 32
Multirotor                    32
Loading Excel: /mnt/storage_11tb/Drive_files_to_syncronize/2 - Patente & Validation/3 -Raw_Patent_Exports_PatSeer_&Gold_Standard/1639__dataset_08_06_26.xlsx


/home/vasco/anaconda3/envs/nb_00b2/lib/python3.10/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/home/vasco/anaconda3/envs/nb_00b2/lib/python3.10/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


  11 patents have no PDF URL (will appear in status report)
  Loaded 1628 patents with valid PDF URLs.

Master export — valid PDF URL: 1628, missing URL: 11


In [2]:
# ── Merge curated labels with real PDF links ────────────────────────────────
merged = arch_df.merge(
    master_df[["Record Number", "pdf_url"]],
    on="Record Number", how="left",
)

unmatched = merged[merged["pdf_url"].isna()]
if not unmatched.empty:
    print(f"⚠ {len(unmatched)} curated patent(s) NOT found in the master export "
          f"(no pdf_url available — excluded from this run):")
    display(unmatched[["Record Number", "Title", "Architecture Category"]])

selected = merged[merged["pdf_url"].notna()].copy().reset_index(drop=True)
print(f"\nMatched patents ready to process: {len(selected)} / {len(merged)}")


⚠ 2 curated patent(s) NOT found in the master export (no pdf_url available — excluded from this run):


,Record Number,Title,Architecture Category
36,WO2024086324A1,AIRCRAFT WITH DUCTED PROPULSION,Ducted Fan Vectored Thrust
60,US2026084845A1,ELECTRIC VERTICAL TAKEOFF AND LANDING AIRCRAFT,Ducted Fan Vectored Thrust



Matched patents ready to process: 126 / 128


In [3]:
# ── Architecture Category → category slug ──────────────────────────────────
# Keys must match the Excel's "Architecture Category" strings exactly.
CATEGORY_SLUGS = {
    "Compound Helicopter":         "compound_helicopter",
    "Ducted Fan Vectored Thrust":  "ducted_fan_vectored_thrust",
    "Lift + Cruise":               "lift_cruise",
    "Multirotor":                  "multirotor",
}

unmapped = sorted(set(selected["Architecture Category"].dropna().unique()) - set(CATEGORY_SLUGS))
assert not unmapped, f"Unmapped Architecture Category values found: {unmapped}"
assert selected["Architecture Category"].notna().all(), "Found rows with a missing Architecture Category"

selected["category"] = selected["Architecture Category"].map(CATEGORY_SLUGS)

print("Category counts (after dropping unmatched patents):")
print(selected["category"].value_counts().to_string())


Category counts (after dropping unmatched patents):
category
compound_helicopter           32
lift_cruise                   32
multirotor                    32
ducted_fan_vectored_thrust    30


In [4]:
import json

# ── 1) logs/selected_patents.json — consumed by get_subset() (subset.mode="selected")
logs_dir = Path(cfg["paths"]["logs"])
logs_dir.mkdir(parents=True, exist_ok=True)

sel_json = {
    "patents": {
        rec: {"selected": True}
        for rec in selected["Record Number"]
    }
}
json_out = logs_dir / "selected_patents.json"
with open(json_out, "w") as f:
    json.dump(sel_json, f, indent=2)
print(f"Saved {len(sel_json['patents'])} patents → {json_out}")

# ── 2) experiment/selected_patents.csv — consumed by organize_processed()
exp_dir = Path(cfg["paths"]["experiment"])
exp_dir.mkdir(parents=True, exist_ok=True)

csv_out = exp_dir / "selected_patents.csv"
selected[["Record Number", "category", "pdf_url"]].to_csv(csv_out, index=False)
print(f"Saved {len(selected)} patents → {csv_out}")


Saved 126 patents → /mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/1639_DS/preliminary analysis v2/data/1639_Patents_logs/selected_patents.json
Saved 126 patents → /mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/1639_DS/preliminary analysis v2/selected_patents.csv


In [5]:
from IPython.display import display

REVIEW_COLS = ["Record Number", "Architecture Category", "category", "Title", "Assignee"]

print("=== Final category counts ===")
print(selected["category"].value_counts().to_string())

for cat, sub in selected.groupby("category"):
    print(f"\n=== {cat} — {len(sub)} patents ===")
    display(sub[REVIEW_COLS].reset_index(drop=True))


=== Final category counts ===
category
compound_helicopter           32
lift_cruise                   32
multirotor                    32
ducted_fan_vectored_thrust    30

=== compound_helicopter — 32 patents ===


,Record Number,Architecture Category,category,Title,Assignee
0,US2022004204A1,Compound Helicopter,compound_helicopter,Aerial Delivery Systems using Unmanned Aircraft,TEXTRON INNOVATIONS INC (US)
1,US2018148160A1,Compound Helicopter,compound_helicopter,ROTOR BLADE HAVING VARIABLE TWIST,BELL HELICOPTER TEXTRON INC (US)
2,CN108408043A,Compound Helicopter,compound_helicopter,Box type tilt rotor aircraft,"BEIHANG UNIV (MUNICIPAL DISTRICT, CN)"
3,CN121448607A,Compound Helicopter,compound_helicopter,Eight-rotor fixed-angle synergistic efficient ...,COOL CENTURY GUANGZHOU INTELLIGENT EQUIPMENT C...
4,US2019039718A1,Compound Helicopter,compound_helicopter,Tiltrotor Aircraft Wings having Buckle Zones,BELL HELICOPTER TEXTRON INC (US)
5,US2024391597A1,Compound Helicopter,compound_helicopter,DAMPING UNIT AS A SECONDARY LOAD PATH FOR A MO...,WISK AERO LLC (US)
6,JP2024519073A,Compound Helicopter,compound_helicopter,Electrical fault isolation in aircraft power d...,LILIUM GMBH
7,US11673662B1,Compound Helicopter,compound_helicopter,Telescoping tail assemblies for use on aircraft,TEXTRON INNOVATIONS INC (US)
8,CN118877199A,Compound Helicopter,compound_helicopter,Electric vertical take-off and landing aircraf...,"LIAO SHIKUI (WUHAN CITY, CN)"
9,WO2024168389A1,Compound Helicopter,compound_helicopter,AN AIRCRAFT WITH A TILTABLE FUSELAGE BODY,BAE SYSTEM AUSTRALIA LTD (AU)



=== ducted_fan_vectored_thrust — 30 patents ===


,Record Number,Architecture Category,category,Title,Assignee
0,US2020062139A1,Ducted Fan Vectored Thrust,ducted_fan_vectored_thrust,AIRCRAFT,PORSCHE AG (DE)
1,US2022177121A1,Ducted Fan Vectored Thrust,ducted_fan_vectored_thrust,LOW NOISE DUCTED FAN,BELL TEXTRON INC (US)
2,US2020239134A1,Ducted Fan Vectored Thrust,ducted_fan_vectored_thrust,HYBRID-ELECTRIC DUCTED FAN TRANSPORT,BELL HELICOPTER TEXTRON INC (US)
3,US2015266571A1,Ducted Fan Vectored Thrust,ducted_fan_vectored_thrust,AERODYNAMICALLY EFFICIENT LIGHTWEIGHT VERTICAL...,JOBY AVIAT INC (US)
4,US2019329882A1,Ducted Fan Vectored Thrust,ducted_fan_vectored_thrust,VARIABLE PITCH ROTOR ASSEMBLY FOR ELECTRICALLY...,AAI CORP (US)
5,US2022355922A1,Ducted Fan Vectored Thrust,ducted_fan_vectored_thrust,VERTICAL TAKE-OFF AND LANDING COCOON-TYPE FLYI...,FILHO ALBERTO CARLOS PEREIRA (BR)
6,DE102018116150A1,Ducted Fan Vectored Thrust,ducted_fan_vectored_thrust,aircraft,PORSCHE AG (DE)
7,US2022144421A1,Ducted Fan Vectored Thrust,ducted_fan_vectored_thrust,ELECTRICALLY POWERED VTOL AIRCRAFT FOR PROVIDI...,AIRSPACE EXPERIENCE TECH INC (US)
8,US9365290B1,Ducted Fan Vectored Thrust,ducted_fan_vectored_thrust,Vertical take off aircraft,MARTIN UAV LLC (US)
9,US2018354614A1,Ducted Fan Vectored Thrust,ducted_fan_vectored_thrust,VERTICAL TAKE-OFF AND LANDING AIRCRAFT,IHI CORP (JP)



=== lift_cruise — 32 patents ===


,Record Number,Architecture Category,category,Title,Assignee
0,CN109263953A,Lift + Cruise,lift_cruise,An aircraft capable of taking off vertically,SHENFENG SCIENCE & TECHNOLOGY OF AVIATION CO L...
1,US2020010187A1,Lift + Cruise,lift_cruise,ELECTRIC POWER SYSTEM ARCHITECTURE AND FAULT T...,JOBY AERO INC (US)
2,UA152017U,Lift + Cruise,lift_cruise,VERTICAL TAKEOFF AND LANDING AIRCRAFT WITH A C...,KONONYKHIN EVGENY ALEXANDROVICH (UA)
3,EP4563464A1,Lift + Cruise,lift_cruise,AIR MOBILITY APPARATUS AND MOBILITY APPARATUS ...,HYUNDAI MOTOR CO (KR); KIA CORP (KR)
4,US2016244158A1,Lift + Cruise,lift_cruise,VERTICAL TAKE-OFF AND LANDING VEHICLE WITH INC...,GOVT OF THE UNITED STATES AS REPRESENTED BY TH...
5,US2022306292A1,Lift + Cruise,lift_cruise,TILTING HEXROTOR AIRCRAFT,BELL TEXTRON INC (US)
6,US6367736B1,Lift + Cruise,lift_cruise,Convertiplane,AGUSTA SPA (US)
7,WO2025065072A1,Lift + Cruise,lift_cruise,CONVERTIBLE AIRCRAFT WITH TANDEM ROTORS THAT A...,FERNANDES DE CASTRO PLINIO (BR)
8,CN106218887A,Lift + Cruise,lift_cruise,Vertical take-off and landing aircraft with di...,XUNYI LTD
9,US2022250742A1,Lift + Cruise,lift_cruise,VERTICAL TAKE-OFF AND LANDING AIRCRAFT WITH AF...,ARCHER AVIATION INC (US)



=== multirotor — 32 patents ===


,Record Number,Architecture Category,category,Title,Assignee
0,US11697495B1,Multirotor,multirotor,Apparatus for guiding a transition between fli...,BETA AIR LLC (US)
1,US2023303243A1,Multirotor,multirotor,DUAL-MOTOR PROPULSION ASSEMBLY,BETA AIR LLC (US)
2,KR102622742B1,Multirotor,multirotor,Hybrid vertical take-off and landing aircraft ...,HAM MYUNG RAE (KR)
3,US2018057155A1,Multirotor,multirotor,MULTICOPTER WITH WIDE SPAN ROTOR CONFIGURATION,KITTY HAWK CORP (US)
4,CN119239928A,Multirotor,multirotor,Vertical take-off and landing flying wing airc...,SHENZHEN SHENKE CHUANG INDUSTRY DEVELOPMENT CO...
5,US2022055743A1,Multirotor,multirotor,FLYING OBJECT,HONDA MOTOR CO LTD (JP)
6,CN114449860A,Multirotor,multirotor,Heat dissipation structure and electric vertic...,SHANGHAI WOLLANT AVIATION TECHNOLOGY CO LTD (M...
7,US11643196B1,Multirotor,multirotor,Teetering propulsor assembly of an electric ve...,BETA AIR LLC (US)
8,US2022048639A1,Multirotor,multirotor,FLYING OBJECT,HONDA MOTOR CO LTD (JP)
9,CN109250104A,Multirotor,multirotor,A vertical take-off and landing aircraft with ...,FOSHAN SHENFENG AVIATION TECH CO LTD
